In [ ]:
 
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
 
 


In [ ]:
# ---------------------------------------------------------------------------
# STEP 1: Create a synthetic imbalanced dataset
# ---------------------------------------------------------------------------
# make_classification is a sklearn helper that generates a fake dataset for
# practicing -- similar in spirit to load_iris, but instead of real flowers,
# it invents random data points with two classes.
X, y = make_classification(
    n_samples=5000,          # total rows
    n_features=10,           # number of measurement columns (like petal length, etc.)
    weights=[0.95, 0.05],    # 95% class 0 (normal), 5% class 1 (fraud) -- this IS the imbalance
    random_state=42
)
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# stratify=y is new -- explained below in the walkthrough
 
#  Imagine you have a bag of 100 marbles: 95 blue, 5 red. You want to split them into two smaller bags. If you just grab handfuls randomly, it's entirely possible one of your smaller bags ends up with 0 red marbles just by unlucky chance (there are so few red ones to begin with). stratify is like being deliberate about it: "make sure each smaller bag gets roughly the same blue-to-red ratio as the original bag" — instead of leaving it to random luck.
# Why does this matter so much specifically for imbalanced data?
# Because if your test set randomly ends up with very few (or zero) minority-class examples by chance, you can't properly evaluate whether your model is actually good at detecting that class — there's barely anything in the test set to check it against. stratify=y protects you from that risk, ensuring your evaluation is meaningful and representative of the real class balance.


In [ ]:
# The problem stratify solves
# Without stratify, train_test_split shuffles and splits your data completely randomly, with no regard for keeping class proportions consistent between the train and test piles. Normally (with balanced data) this is totally fine — random chance will naturally produce roughly similar proportions in both piles anyway.
# But with imbalanced data, this randomness becomes risky. Remember, our fraud class is only ~5% of the data (~250 rows total out of 5,000). If you split randomly without stratifying, it's entirely possible — just by bad luck — that the test set ends up with, say, only 30 fraud examples instead of the expected ~50 (20% of 250), or the training set ends up with a slightly different fraud ratio than the test set. With such a small minority class to begin with, random chance can meaningfully skew things.
# What stratify=y actually does
# It tells train_test_split: "Whatever the class proportions are in the full dataset, preserve those exact same proportions in both the train pile AND the test pile."
# So if the full dataset is 95% class 0 / 5% class 1, stratify=y guarantees that:

# The training pile will also be ~95% class 0 / ~5% class 1
# The test pile will also be ~95% class 0 / ~5% class 1

# Instead of leaving those proportions up to random chance, it deliberately mirrors the original ratio in both pieces.
# Simple analogy
# Imagine you have a bag of 100 marbles: 95 blue, 5 red. You want to split them into two smaller bags. If you just grab handfuls randomly, it's entirely possible one of your smaller bags ends up with 0 red marbles just by unlucky chance (there are so few red ones to begin with). stratify is like being deliberate about it: "make sure each smaller bag gets roughly the same blue-to-red ratio as the original bag" — instead of leaving it to random luck.
# Why does this matter so much specifically for imbalanced data?
# Because if your test set randomly ends up with very few (or zero) minority-class examples by chance, you can't properly evaluate whether your model is actually good at detecting that class — there's barely anything in the test set to check it against. stratify=y protects you from that risk, ensuring your evaluation is meaningful and representative of the real class balance.
# Why didn't we need stratify back in the Iris/Optuna example?
# Because Iris is a small, but balanced dataset (50/50/50 across 3 species) — random splitting was very unlikely to meaningfully distort those proportions. It's really only imbalanced data (or very small datasets) where stratify becomes an important habit.

In [ ]:
# ---------------------------------------------------------------------------
# STEP 2: Train a model the NAIVE way (no imbalance handling at all)
# ---------------------------------------------------------------------------
naive_model = LogisticRegression(random_state=42)
naive_model.fit(X_train, y_train)
naive_pred = naive_model.predict(X_test)
 
print("=== NAIVE MODEL (no imbalance handling) ===")
print(confusion_matrix(y_test, naive_pred))
print(classification_report(y_test, naive_pred))
 
 


In [ ]:
# ---------------------------------------------------------------------------
# STEP 3: Train the SAME model, but with class_weight="balanced"
# ---------------------------------------------------------------------------
weighted_model = LogisticRegression(class_weight="balanced", random_state=42)
weighted_model.fit(X_train, y_train)
weighted_pred = weighted_model.predict(X_test)
 
print("=== WEIGHTED MODEL (class_weight='balanced') ===")
print(confusion_matrix(y_test, weighted_pred))
print(classification_report(y_test, weighted_pred))